In [ ]:

# 1. Check GPU available, print torch/CUDA version (T4 accelerator assumed enabled).
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA Version:", torch.version.cuda)
    print("Device Name:", torch.cuda.get_device_name(0))


In [ ]:

# 2. Set TORCH_CUDA_ARCH_LIST="7.5" (T4 = Turing).
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
print("Set TORCH_CUDA_ARCH_LIST to", os.environ["TORCH_CUDA_ARCH_LIST"])


In [ ]:


!git clone --recursive "https://github.com/vandimmi/Ponyo---BTS-Digital-Twin.git" /kaggle/working/repo
import os
os.chdir('/kaggle/working/repo')
print("Changed directory to:", os.getcwd())


In [ ]:

# 4. Install requirements.txt, then build diff-gaussian-rasterization and simple-knn from source.
!pip install -q -r requirements.txt
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn
print("Installed CUDA extensions successfully!")


In [ ]:
%%writefile render_test_poses.py
import os
import torch
import csv
from scene import Scene
from gaussian_renderer import render
from argparse import ArgumentParser
from utils.system_utils import mkdir_p
from torchvision.utils import save_image
from test_pose_loader import TestCamera, focal2fov

def main():
    parser = ArgumentParser(description="Render test poses from test_poses.csv")
    parser.add_argument("-m", "--model_path", type=str, required=True)
    parser.add_argument("--test_poses_csv", type=str, required=True)
    parser.add_argument("--output_dir", type=str, default="renders")
    args = parser.parse_args()

    # Load checkpoint
    # (Assuming we have GaussianModel instantiated; simplified for demonstration)
    from scene.gaussian_model import GaussianModel
    gaussians = GaussianModel(sh_degree=3)
    
    # Normally we load the scene/checkpoint here
    print(f"Loading checkpoint from {args.model_path}")
    # gaussians.load_ply(...)
    
    mkdir_p(args.output_dir)
    
    # Read CSV
    with open(args.test_poses_csv, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            image_name, qw, qx, qy, qz, tx, ty, tz, fx, fy, cx, cy, width, height = row
            
            # Convert COLMAP qvec/tvec to R, T
            # R should be World-to-Camera, transposed for 3DGS
            qw, qx, qy, qz = map(float, (qw, qx, qy, qz))
            tx, ty, tz = map(float, (tx, ty, tz))
            
            R = np.array([
                [1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qz * qw), 2 * (qx * qz + qy * qw)],
                [2 * (qx * qy + qz * qw), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qx * qw)],
                [2 * (qx * qz - qy * qw), 2 * (qy * qz + qx * qw), 1 - 2 * (qx**2 + qy**2)]
            ])
            Rt = np.transpose(R)
            T = np.array([tx, ty, tz])
            
            FovY = focal2fov(float(fy), float(height))
            FovX = focal2fov(float(fx), float(width))
            
            cam = TestCamera(uid=0, R=Rt, T=T, FovX=FovX, FovY=FovY, 
                             image_name=image_name, width=int(width), height=int(height))
            
            # rendering call
            print(f"Rendering {image_name}")
            # render_pkg = render(cam, gaussians, pipeline_args, background)
            # image = render_pkg["render"]
            # save_image(image, os.path.join(args.output_dir, image_name))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile test_pose_loader.py
import os
import sys
import torch
import numpy as np
import math
from utils.graphics_utils import focal2fov

def colmap_pose_to_w2c(qw, qx, qy, qz, tx, ty, tz):
    """
    Convert a COLMAP test pose (which represents World-to-Camera transform)
    to a PyTorch tensor format expected by 3DGS.
    """
    # Quaternion to rotation matrix (World-to-Camera)
    # COLMAP convention: (qw, qx, qy, qz)
    R = np.array([
        [1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qz * qw), 2 * (qx * qz + qy * qw)],
        [2 * (qx * qy + qz * qw), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qx * qw)],
        [2 * (qx * qz - qy * qw), 2 * (qy * qz + qx * qw), 1 - 2 * (qx**2 + qy**2)]
    ])
    
    T = np.array([tx, ty, tz])
    
    # In 3DGS, the final world_view_transform is the transpose of the 4x4 Rt matrix
    Rt = np.zeros((4, 4))
    Rt[:3, :3] = R
    Rt[:3, 3] = T
    Rt[3, 3] = 1.0
    
    world_view_transform = torch.tensor(Rt).float().transpose(0, 1)
    
    return world_view_transform

def get_projection_matrix(znear, zfar, fovX, fovY):
    tanHalfFovY = math.tan((fovY / 2))
    tanHalfFovX = math.tan((fovX / 2))

    top = tanHalfFovY * znear
    bottom = -top
    right = tanHalfFovX * znear
    left = -right

    P = torch.zeros(4, 4)

    z_sign = 1.0

    P[0, 0] = 2.0 * znear / (right - left)
    P[1, 1] = 2.0 * znear / (top - bottom)
    P[0, 2] = (right + left) / (right - left)
    P[1, 2] = (top + bottom) / (top - bottom)
    P[3, 2] = z_sign
    P[2, 2] = z_sign * zfar / (zfar - znear)
    P[2, 3] = -(zfar * znear) / (zfar - znear)
    return P

class TestCamera(torch.nn.Module):
    def __init__(self, uid, R, T, FovX, FovY, image_name, width, height):
        super(TestCamera, self).__init__()
        self.uid = uid
        self.image_name = image_name
        self.image_width = width
        self.image_height = height
        
        # We need world_view_transform and full_proj_transform
        # R is expected transposed
        self.world_view_transform = torch.tensor(np.column_stack((R, T))).float().cuda()
        self.world_view_transform = torch.cat([self.world_view_transform, torch.tensor([[0,0,0,1]]).float().cuda()], dim=0)
        
        self.projection_matrix = get_projection_matrix(znear=0.01, zfar=100.0, fovX=FovX, fovY=FovY).transpose(0, 1).cuda()
        self.full_proj_transform = (self.world_view_transform.unsqueeze(0).bmm(self.projection_matrix.unsqueeze(0))).squeeze(0)
        self.camera_center = self.world_view_transform.inverse()[3, :3]


In [ ]:
%%writefile package_submission.py
import os
import zipfile
import shutil

def package_submission():
    renders_dir = "renders"
    output_zip = "submission.zip"
    
    if not os.path.exists(renders_dir):
        print(f"Error: {renders_dir} not found. Render the test poses first.")
        return

    print("Creating submission.zip...")
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # We need to package it as scene_xxx/000x.png
        # Let's say the scene is HCM0421
        scene_name = "scene_HCM0421"
        for i, f in enumerate(sorted(os.listdir(renders_dir))):
            if f.endswith('.png') or f.endswith('.jpg'):
                # Rename to 0000.png, 0001.png etc.
                arcname = f"{scene_name}/{i:04d}.png"
                file_path = os.path.join(renders_dir, f)
                zipf.write(file_path, arcname)
                print(f"Added {file_path} -> {arcname}")
                
    print(f"Submission packaged into {output_zip}")

if __name__ == "__main__":
    package_submission()


In [ ]:
%%writefile eval.py
import os
import argparse
import torch
import lpips
import numpy as np
from PIL import Image
from math import log10
from pytorch_msssim import ssim

def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    PIXEL_MAX = 1.0
    return 20 * log10(PIXEL_MAX / torch.sqrt(mse))

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--renders_dir", type=str, required=True, help="Directory with rendered images")
    parser.add_argument("--gt_dir", type=str, required=True, help="Directory with ground truth images")
    parser.add_argument("--psnr_max", type=float, default=35.0, help="Max PSNR for normalization")
    args = parser.parse_args()

    # VGG is the standard backbone used in NeRF, Mip-NeRF 360, and 3DGS benchmarks
    # because it correlates better with human perception of structural changes in novel views
    # compared to AlexNet or SqueezeNet.
    loss_fn_vgg = lpips.LPIPS(net='vgg').cuda()

    render_files = sorted([f for f in os.listdir(args.renders_dir) if f.endswith(('.png', '.jpg'))])
    
    total_psnr = 0.0
    total_ssim = 0.0
    total_lpips = 0.0
    count = 0

    for f in render_files:
        render_path = os.path.join(args.renders_dir, f)
        gt_path = os.path.join(args.gt_dir, f)
        
        if not os.path.exists(gt_path):
            print(f"Warning: GT image not found for {f}, skipping.")
            continue

        render_img = np.array(Image.open(render_path).convert("RGB")) / 255.0
        gt_img = np.array(Image.open(gt_path).convert("RGB")) / 255.0

        # Convert to BCHW PyTorch tensors
        render_tensor = torch.from_numpy(render_img).permute(2, 0, 1).unsqueeze(0).float().cuda()
        gt_tensor = torch.from_numpy(gt_img).permute(2, 0, 1).unsqueeze(0).float().cuda()

        # Calculate PSNR
        psnr_val = calculate_psnr(render_tensor, gt_tensor)
        
        # Calculate SSIM
        ssim_val = ssim(render_tensor, gt_tensor, data_range=1.0, size_average=True).item()
        
        # Calculate LPIPS (requires range [-1, 1])
        render_tensor_lpips = render_tensor * 2.0 - 1.0
        gt_tensor_lpips = gt_tensor * 2.0 - 1.0
        lpips_val = loss_fn_vgg(render_tensor_lpips, gt_tensor_lpips).item()

        total_psnr += psnr_val
        total_ssim += ssim_val
        total_lpips += lpips_val
        count += 1

    if count == 0:
        print("No matching images found for evaluation.")
        return

    avg_psnr = total_psnr / count
    avg_ssim = total_ssim / count
    avg_lpips = total_lpips / count

    # Normalize PSNR
    psnr_norm = max(0.0, min(avg_psnr / args.psnr_max, 1.0))
    
    # Calculate Final Score
    score = 0.4 * (1 - avg_lpips) + 0.3 * avg_ssim + 0.3 * psnr_norm

    print("--- Evaluation Results ---")
    print(f"Images Evaluated: {count}")
    print(f"Average PSNR: {avg_psnr:.4f}")
    print(f"Average SSIM: {avg_ssim:.4f}")
    print(f"Average LPIPS (VGG): {avg_lpips:.4f}")
    print(f"PSNR Norm (max={args.psnr_max}): {psnr_norm:.4f}")
    print(f"Final Score: {score:.4f}")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
#
# Copyright (C) 2023, Inria
# GRAPHDECO research group, https://team.inria.fr/graphdeco
# All rights reserved.
#
# This software is free for non-commercial, research and evaluation use 
# under the terms of the LICENSE.md file.
#
# For inquiries contact  george.drettakis@inria.fr
#

import os
import torch
from random import randint
from utils.loss_utils import l1_loss, ssim
from gaussian_renderer import render, network_gui
import sys
from scene import Scene, GaussianModel
from utils.general_utils import safe_state, get_expon_lr_func
import uuid
from tqdm import tqdm
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_FOUND = True
except ImportError:
    TENSORBOARD_FOUND = False

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except:
    FUSED_SSIM_AVAILABLE = False

try:
    from diff_gaussian_rasterization import SparseGaussianAdam
    SPARSE_ADAM_AVAILABLE = True
except:
    SPARSE_ADAM_AVAILABLE = False

def training(dataset, opt, pipe, testing_iterations, saving_iterations, checkpoint_iterations, checkpoint, debug_from):

    if not SPARSE_ADAM_AVAILABLE and opt.optimizer_type == "sparse_adam":
        sys.exit(f"Trying to use sparse adam but it is not installed, please install the correct rasterizer using pip install [3dgs_accel].")

    first_iter = 0
    tb_writer = prepare_output_and_logger(dataset)
    gaussians = GaussianModel(dataset.sh_degree, opt.optimizer_type)
    scene = Scene(dataset, gaussians)
    gaussians.training_setup(opt)
    if checkpoint:
        (model_params, first_iter) = torch.load(checkpoint)
        gaussians.restore(model_params, opt)

    bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
    background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

    iter_start = torch.cuda.Event(enable_timing = True)
    iter_end = torch.cuda.Event(enable_timing = True)

    use_sparse_adam = opt.optimizer_type == "sparse_adam" and SPARSE_ADAM_AVAILABLE 
    depth_l1_weight = get_expon_lr_func(opt.depth_l1_weight_init, opt.depth_l1_weight_final, max_steps=opt.iterations)

    viewpoint_stack = scene.getTrainCameras().copy()
    viewpoint_indices = list(range(len(viewpoint_stack)))
    ema_loss_for_log = 0.0
    ema_Ll1depth_for_log = 0.0

    progress_bar = tqdm(range(first_iter, opt.iterations), desc="Training progress")
    first_iter += 1
    for iteration in range(first_iter, opt.iterations + 1):
        if network_gui.conn == None:
            network_gui.try_connect()
        while network_gui.conn != None:
            try:
                net_image_bytes = None
                custom_cam, do_training, pipe.convert_SHs_python, pipe.compute_cov3D_python, keep_alive, scaling_modifer = network_gui.receive()
                if custom_cam != None:
                    net_image = render(custom_cam, gaussians, pipe, background, scaling_modifier=scaling_modifer, use_trained_exp=dataset.train_test_exp, separate_sh=SPARSE_ADAM_AVAILABLE)["render"]
                    net_image_bytes = memoryview((torch.clamp(net_image, min=0, max=1.0) * 255).byte().permute(1, 2, 0).contiguous().cpu().numpy())
                network_gui.send(net_image_bytes, dataset.source_path)
                if do_training and ((iteration < int(opt.iterations)) or not keep_alive):
                    break
            except Exception as e:
                network_gui.conn = None

        iter_start.record()

        gaussians.update_learning_rate(iteration)
        if iteration % 1000 == 0:
            print(f"\n[ITER {iteration}] Peak VRAM: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")


        # Every 1000 its we increase the levels of SH up to a maximum degree
        if iteration % 1000 == 0:
            gaussians.oneupSHdegree()

        # Pick a random Camera
        if not viewpoint_stack:
            viewpoint_stack = scene.getTrainCameras().copy()
            viewpoint_indices = list(range(len(viewpoint_stack)))
        rand_idx = randint(0, len(viewpoint_indices) - 1)
        viewpoint_cam = viewpoint_stack.pop(rand_idx)
        vind = viewpoint_indices.pop(rand_idx)

        # Render
        if (iteration - 1) == debug_from:
            pipe.debug = True

        bg = torch.rand((3), device="cuda") if opt.random_background else background

        render_pkg = render(viewpoint_cam, gaussians, pipe, bg, use_trained_exp=dataset.train_test_exp, separate_sh=SPARSE_ADAM_AVAILABLE)
        image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]

        if viewpoint_cam.alpha_mask is not None:
            alpha_mask = viewpoint_cam.alpha_mask.cuda()
            image *= alpha_mask

        # Loss
        gt_image = viewpoint_cam.original_image.cuda()
        Ll1 = l1_loss(image, gt_image)
        if FUSED_SSIM_AVAILABLE:
            ssim_value = fused_ssim(image.unsqueeze(0), gt_image.unsqueeze(0))
        else:
            ssim_value = ssim(image, gt_image)

        loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim_value)

        # Depth regularization
        Ll1depth_pure = 0.0
        if depth_l1_weight(iteration) > 0 and viewpoint_cam.depth_reliable:
            invDepth = render_pkg["depth"]
            mono_invdepth = viewpoint_cam.invdepthmap.cuda()
            depth_mask = viewpoint_cam.depth_mask.cuda()

            Ll1depth_pure = torch.abs((invDepth  - mono_invdepth) * depth_mask).mean()
            Ll1depth = depth_l1_weight(iteration) * Ll1depth_pure 
            loss += Ll1depth
            Ll1depth = Ll1depth.item()
        else:
            Ll1depth = 0

        loss.backward()

        iter_end.record()

        with torch.no_grad():
            # Progress bar
            ema_loss_for_log = 0.4 * loss.item() + 0.6 * ema_loss_for_log
            ema_Ll1depth_for_log = 0.4 * Ll1depth + 0.6 * ema_Ll1depth_for_log

            if iteration % 10 == 0:
                progress_bar.set_postfix({"Loss": f"{ema_loss_for_log:.{7}f}", "Depth Loss": f"{ema_Ll1depth_for_log:.{7}f}"})
                progress_bar.update(10)
            if iteration == opt.iterations:
                progress_bar.close()

            # Log and save
            training_report(tb_writer, iteration, Ll1, loss, l1_loss, iter_start.elapsed_time(iter_end), testing_iterations, scene, render, (pipe, background, 1., SPARSE_ADAM_AVAILABLE, None, dataset.train_test_exp), dataset.train_test_exp)
            if (iteration in saving_iterations):
                print("\n[ITER {}] Saving Gaussians".format(iteration))
                scene.save(iteration)

            # Densification
            if iteration < opt.densify_until_iter:
                # Keep track of max radii in image-space for pruning
                gaussians.max_radii2D[visibility_filter] = torch.max(gaussians.max_radii2D[visibility_filter], radii[visibility_filter])
                gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter)

                if iteration > opt.densify_from_iter and iteration % opt.densification_interval == 0:
                    size_threshold = 20 if iteration > opt.opacity_reset_interval else None
                    gaussians.densify_and_prune(opt.densify_grad_threshold, 0.005, scene.cameras_extent, size_threshold, radii)
                
                if iteration % opt.opacity_reset_interval == 0 or (dataset.white_background and iteration == opt.densify_from_iter):
                    gaussians.reset_opacity()

            # Optimizer step
            if iteration < opt.iterations:
                gaussians.exposure_optimizer.step()
                gaussians.exposure_optimizer.zero_grad(set_to_none = True)
                if use_sparse_adam:
                    visible = radii > 0
                    gaussians.optimizer.step(visible, radii.shape[0])
                    gaussians.optimizer.zero_grad(set_to_none = True)
                else:
                    gaussians.optimizer.step()
                    gaussians.optimizer.zero_grad(set_to_none = True)

            if (iteration in checkpoint_iterations):
                print("\\n[ITER {}] Saving Checkpoint".format(iteration))
                save_path = scene.model_path + "/chkpnt" + str(iteration) + ".pth"
                torch.save((gaussians.capture(), iteration), save_path)
                import shutil
                if os.path.exists(os.environ.get('KAGGLE_WORKING_DIR', '/kaggle/working/')):
                    shutil.copy(save_path, os.environ.get('KAGGLE_WORKING_DIR', '/kaggle/working/'))

def prepare_output_and_logger(args):    
    if not args.model_path:
        if os.getenv('OAR_JOB_ID'):
            unique_str=os.getenv('OAR_JOB_ID')
        else:
            unique_str = str(uuid.uuid4())
        args.model_path = os.path.join("./output/", unique_str[0:10])
        
    # Set up output folder
    print("Output folder: {}".format(args.model_path))
    os.makedirs(args.model_path, exist_ok = True)
    with open(os.path.join(args.model_path, "cfg_args"), 'w') as cfg_log_f:
        cfg_log_f.write(str(Namespace(**vars(args))))

    # Create Tensorboard writer
    tb_writer = None
    if TENSORBOARD_FOUND:
        tb_writer = SummaryWriter(args.model_path)
    else:
        print("Tensorboard not available: not logging progress")
    return tb_writer

def training_report(tb_writer, iteration, Ll1, loss, l1_loss, elapsed, testing_iterations, scene : Scene, renderFunc, renderArgs, train_test_exp):
    if tb_writer:
        tb_writer.add_scalar('train_loss_patches/l1_loss', Ll1.item(), iteration)
        tb_writer.add_scalar('train_loss_patches/total_loss', loss.item(), iteration)
        tb_writer.add_scalar('iter_time', elapsed, iteration)

    # Report test and samples of training set
    if iteration in testing_iterations:
        torch.cuda.empty_cache()
        validation_configs = ({'name': 'test', 'cameras' : scene.getTestCameras()}, 
                              {'name': 'train', 'cameras' : [scene.getTrainCameras()[idx % len(scene.getTrainCameras())] for idx in range(5, 30, 5)]})

        for config in validation_configs:
            if config['cameras'] and len(config['cameras']) > 0:
                l1_test = 0.0
                psnr_test = 0.0
                for idx, viewpoint in enumerate(config['cameras']):
                    image = torch.clamp(renderFunc(viewpoint, scene.gaussians, *renderArgs)["render"], 0.0, 1.0)
                    gt_image = torch.clamp(viewpoint.original_image.to("cuda"), 0.0, 1.0)
                    if train_test_exp:
                        image = image[..., image.shape[-1] // 2:]
                        gt_image = gt_image[..., gt_image.shape[-1] // 2:]
                    if tb_writer and (idx < 5):
                        tb_writer.add_images(config['name'] + "_view_{}/render".format(viewpoint.image_name), image[None], global_step=iteration)
                        if iteration == testing_iterations[0]:
                            tb_writer.add_images(config['name'] + "_view_{}/ground_truth".format(viewpoint.image_name), gt_image[None], global_step=iteration)
                    l1_test += l1_loss(image, gt_image).mean().double()
                    psnr_test += psnr(image, gt_image).mean().double()
                psnr_test /= len(config['cameras'])
                l1_test /= len(config['cameras'])          
                print("\n[ITER {}] Evaluating {}: L1 {} PSNR {}".format(iteration, config['name'], l1_test, psnr_test))
                if tb_writer:
                    tb_writer.add_scalar(config['name'] + '/loss_viewpoint - l1_loss', l1_test, iteration)
                    tb_writer.add_scalar(config['name'] + '/loss_viewpoint - psnr', psnr_test, iteration)

        if tb_writer:
            tb_writer.add_histogram("scene/opacity_histogram", scene.gaussians.get_opacity, iteration)
            tb_writer.add_scalar('total_points', scene.gaussians.get_xyz.shape[0], iteration)
        torch.cuda.empty_cache()

if __name__ == "__main__":
    # Set up command line argument parser
    parser = ArgumentParser(description="Training script parameters")
    lp = ModelParams(parser)
    op = OptimizationParams(parser)
    pp = PipelineParams(parser)
    parser.add_argument('--ip', type=str, default="127.0.0.1")
    parser.add_argument('--port', type=int, default=6009)
    parser.add_argument('--debug_from', type=int, default=-1)
    parser.add_argument('--detect_anomaly', action='store_true', default=False)
    parser.add_argument("--test_iterations", nargs="+", type=int, default=[7_000, 30_000])
    parser.add_argument("--save_iterations", nargs="+", type=int, default=[7_000, 30_000])
    parser.add_argument("--quiet", action="store_true")
    parser.add_argument('--disable_viewer', action='store_true', default=False)
    parser.add_argument("--checkpoint_iterations", nargs="+", type=int, default=[])
    parser.add_argument("--start_checkpoint", type=str, default = None)
    parser.add_argument("--kaggle_working_dir", type=str, default='/kaggle/working/')
    parser.add_argument("--resume", action='store_true', help='Auto-resume from latest checkpoint in model_path')
    args = parser.parse_args(sys.argv[1:])
    args.save_iterations.append(args.iterations)
    
    
    # Resume logic
    if args.resume and not args.start_checkpoint:
        import glob
        chkpts = glob.glob(os.path.join(args.model_path, "chkpnt*.pth"))
        if len(chkpts) > 0:
            # Sort by iteration number
            chkpts.sort(key=lambda x: int(re.findall(r"chkpnt(\d+).pth", x)[0]))
            args.start_checkpoint = chkpts[-1]
            print(f"Resuming from {args.start_checkpoint}")

    print("Optimizing " + args.model_path)

    # Initialize system state (RNG)
    safe_state(args.quiet)

    # Start GUI server, configure and run training
    if not args.disable_viewer:
        network_gui.init(args.ip, args.port)
    torch.autograd.set_detect_anomaly(args.detect_anomaly)
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)

    # All done
    print("\nTraining complete.")
    if os.path.exists(args.kaggle_working_dir):
        import zipfile
        out_zip = os.path.join(args.kaggle_working_dir, "training_run.zip")
        print(f"Zipping outputs to {out_zip}")
        with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(args.model_path):
                for f in files:
                    # Zip everything in the output directory
                    abs_path = os.path.join(root, f)
                    rel_path = os.path.relpath(abs_path, args.model_path)
                    zipf.write(abs_path, rel_path)


In [ ]:
%%writefile render_test_poses.py
import os
import torch
import csv
from scene import Scene
from gaussian_renderer import render
from argparse import ArgumentParser
from utils.system_utils import mkdir_p
from torchvision.utils import save_image
from test_pose_loader import TestCamera, focal2fov

def main():
    parser = ArgumentParser(description="Render test poses from test_poses.csv")
    parser.add_argument("-m", "--model_path", type=str, required=True)
    parser.add_argument("--test_poses_csv", type=str, required=True)
    parser.add_argument("--output_dir", type=str, default="renders")
    args = parser.parse_args()

    # Load checkpoint
    # (Assuming we have GaussianModel instantiated; simplified for demonstration)
    from scene.gaussian_model import GaussianModel
    gaussians = GaussianModel(sh_degree=3)
    
    # Normally we load the scene/checkpoint here
    print(f"Loading checkpoint from {args.model_path}")
    # gaussians.load_ply(...)
    
    mkdir_p(args.output_dir)
    
    # Read CSV
    with open(args.test_poses_csv, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)
        for row in reader:
            image_name, qw, qx, qy, qz, tx, ty, tz, fx, fy, cx, cy, width, height = row
            
            # Convert COLMAP qvec/tvec to R, T
            # R should be World-to-Camera, transposed for 3DGS
            qw, qx, qy, qz = map(float, (qw, qx, qy, qz))
            tx, ty, tz = map(float, (tx, ty, tz))
            
            R = np.array([
                [1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qz * qw), 2 * (qx * qz + qy * qw)],
                [2 * (qx * qy + qz * qw), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qx * qw)],
                [2 * (qx * qz - qy * qw), 2 * (qy * qz + qx * qw), 1 - 2 * (qx**2 + qy**2)]
            ])
            Rt = np.transpose(R)
            T = np.array([tx, ty, tz])
            
            FovY = focal2fov(float(fy), float(height))
            FovX = focal2fov(float(fx), float(width))
            
            cam = TestCamera(uid=0, R=Rt, T=T, FovX=FovX, FovY=FovY, 
                             image_name=image_name, width=int(width), height=int(height))
            
            # rendering call
            print(f"Rendering {image_name}")
            # render_pkg = render(cam, gaussians, pipeline_args, background)
            # image = render_pkg["render"]
            # save_image(image, os.path.join(args.output_dir, image_name))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile test_pose_loader.py
import os
import sys
import torch
import numpy as np
import math
from utils.graphics_utils import focal2fov

def colmap_pose_to_w2c(qw, qx, qy, qz, tx, ty, tz):
    """
    Convert a COLMAP test pose (which represents World-to-Camera transform)
    to a PyTorch tensor format expected by 3DGS.
    """
    # Quaternion to rotation matrix (World-to-Camera)
    # COLMAP convention: (qw, qx, qy, qz)
    R = np.array([
        [1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qz * qw), 2 * (qx * qz + qy * qw)],
        [2 * (qx * qy + qz * qw), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qx * qw)],
        [2 * (qx * qz - qy * qw), 2 * (qy * qz + qx * qw), 1 - 2 * (qx**2 + qy**2)]
    ])
    
    T = np.array([tx, ty, tz])
    
    # In 3DGS, the final world_view_transform is the transpose of the 4x4 Rt matrix
    Rt = np.zeros((4, 4))
    Rt[:3, :3] = R
    Rt[:3, 3] = T
    Rt[3, 3] = 1.0
    
    world_view_transform = torch.tensor(Rt).float().transpose(0, 1)
    
    return world_view_transform

def get_projection_matrix(znear, zfar, fovX, fovY):
    tanHalfFovY = math.tan((fovY / 2))
    tanHalfFovX = math.tan((fovX / 2))

    top = tanHalfFovY * znear
    bottom = -top
    right = tanHalfFovX * znear
    left = -right

    P = torch.zeros(4, 4)

    z_sign = 1.0

    P[0, 0] = 2.0 * znear / (right - left)
    P[1, 1] = 2.0 * znear / (top - bottom)
    P[0, 2] = (right + left) / (right - left)
    P[1, 2] = (top + bottom) / (top - bottom)
    P[3, 2] = z_sign
    P[2, 2] = z_sign * zfar / (zfar - znear)
    P[2, 3] = -(zfar * znear) / (zfar - znear)
    return P

class TestCamera(torch.nn.Module):
    def __init__(self, uid, R, T, FovX, FovY, image_name, width, height):
        super(TestCamera, self).__init__()
        self.uid = uid
        self.image_name = image_name
        self.image_width = width
        self.image_height = height
        
        # We need world_view_transform and full_proj_transform
        # R is expected transposed
        self.world_view_transform = torch.tensor(np.column_stack((R, T))).float().cuda()
        self.world_view_transform = torch.cat([self.world_view_transform, torch.tensor([[0,0,0,1]]).float().cuda()], dim=0)
        
        self.projection_matrix = get_projection_matrix(znear=0.01, zfar=100.0, fovX=FovX, fovY=FovY).transpose(0, 1).cuda()
        self.full_proj_transform = (self.world_view_transform.unsqueeze(0).bmm(self.projection_matrix.unsqueeze(0))).squeeze(0)
        self.camera_center = self.world_view_transform.inverse()[3, :3]


In [ ]:
%%writefile package_submission.py
import os
import zipfile
import shutil

def package_submission():
    renders_dir = "renders"
    output_zip = "submission.zip"
    
    if not os.path.exists(renders_dir):
        print(f"Error: {renders_dir} not found. Render the test poses first.")
        return

    print("Creating submission.zip...")
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # We need to package it as scene_xxx/000x.png
        # Let's say the scene is HCM0421
        scene_name = "scene_HCM0421"
        for i, f in enumerate(sorted(os.listdir(renders_dir))):
            if f.endswith('.png') or f.endswith('.jpg'):
                # Rename to 0000.png, 0001.png etc.
                arcname = f"{scene_name}/{i:04d}.png"
                file_path = os.path.join(renders_dir, f)
                zipf.write(file_path, arcname)
                print(f"Added {file_path} -> {arcname}")
                
    print(f"Submission packaged into {output_zip}")

if __name__ == "__main__":
    package_submission()


In [ ]:
%%writefile eval.py
import os
import argparse
import torch
import lpips
import numpy as np
from PIL import Image
from math import log10
from pytorch_msssim import ssim

def calculate_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    PIXEL_MAX = 1.0
    return 20 * log10(PIXEL_MAX / torch.sqrt(mse))

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--renders_dir", type=str, required=True, help="Directory with rendered images")
    parser.add_argument("--gt_dir", type=str, required=True, help="Directory with ground truth images")
    parser.add_argument("--psnr_max", type=float, default=35.0, help="Max PSNR for normalization")
    args = parser.parse_args()

    # VGG is the standard backbone used in NeRF, Mip-NeRF 360, and 3DGS benchmarks
    # because it correlates better with human perception of structural changes in novel views
    # compared to AlexNet or SqueezeNet.
    loss_fn_vgg = lpips.LPIPS(net='vgg').cuda()

    render_files = sorted([f for f in os.listdir(args.renders_dir) if f.endswith(('.png', '.jpg'))])
    
    total_psnr = 0.0
    total_ssim = 0.0
    total_lpips = 0.0
    count = 0

    for f in render_files:
        render_path = os.path.join(args.renders_dir, f)
        gt_path = os.path.join(args.gt_dir, f)
        
        if not os.path.exists(gt_path):
            print(f"Warning: GT image not found for {f}, skipping.")
            continue

        render_img = np.array(Image.open(render_path).convert("RGB")) / 255.0
        gt_img = np.array(Image.open(gt_path).convert("RGB")) / 255.0

        # Convert to BCHW PyTorch tensors
        render_tensor = torch.from_numpy(render_img).permute(2, 0, 1).unsqueeze(0).float().cuda()
        gt_tensor = torch.from_numpy(gt_img).permute(2, 0, 1).unsqueeze(0).float().cuda()

        # Calculate PSNR
        psnr_val = calculate_psnr(render_tensor, gt_tensor)
        
        # Calculate SSIM
        ssim_val = ssim(render_tensor, gt_tensor, data_range=1.0, size_average=True).item()
        
        # Calculate LPIPS (requires range [-1, 1])
        render_tensor_lpips = render_tensor * 2.0 - 1.0
        gt_tensor_lpips = gt_tensor * 2.0 - 1.0
        lpips_val = loss_fn_vgg(render_tensor_lpips, gt_tensor_lpips).item()

        total_psnr += psnr_val
        total_ssim += ssim_val
        total_lpips += lpips_val
        count += 1

    if count == 0:
        print("No matching images found for evaluation.")
        return

    avg_psnr = total_psnr / count
    avg_ssim = total_ssim / count
    avg_lpips = total_lpips / count

    # Normalize PSNR
    psnr_norm = max(0.0, min(avg_psnr / args.psnr_max, 1.0))
    
    # Calculate Final Score
    score = 0.4 * (1 - avg_lpips) + 0.3 * avg_ssim + 0.3 * psnr_norm

    print("--- Evaluation Results ---")
    print(f"Images Evaluated: {count}")
    print(f"Average PSNR: {avg_psnr:.4f}")
    print(f"Average SSIM: {avg_ssim:.4f}")
    print(f"Average LPIPS (VGG): {avg_lpips:.4f}")
    print(f"PSNR Norm (max={args.psnr_max}): {psnr_norm:.4f}")
    print(f"Final Score: {score:.4f}")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
#
# Copyright (C) 2023, Inria
# GRAPHDECO research group, https://team.inria.fr/graphdeco
# All rights reserved.
#
# This software is free for non-commercial, research and evaluation use 
# under the terms of the LICENSE.md file.
#
# For inquiries contact  george.drettakis@inria.fr
#

import os
import torch
from random import randint
from utils.loss_utils import l1_loss, ssim
from gaussian_renderer import render, network_gui
import sys
from scene import Scene, GaussianModel
from utils.general_utils import safe_state, get_expon_lr_func
import uuid
from tqdm import tqdm
from utils.image_utils import psnr
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams
try:
    from torch.utils.tensorboard import SummaryWriter
    TENSORBOARD_FOUND = True
except ImportError:
    TENSORBOARD_FOUND = False

try:
    from fused_ssim import fused_ssim
    FUSED_SSIM_AVAILABLE = True
except:
    FUSED_SSIM_AVAILABLE = False

try:
    from diff_gaussian_rasterization import SparseGaussianAdam
    SPARSE_ADAM_AVAILABLE = True
except:
    SPARSE_ADAM_AVAILABLE = False

def training(dataset, opt, pipe, testing_iterations, saving_iterations, checkpoint_iterations, checkpoint, debug_from):

    if not SPARSE_ADAM_AVAILABLE and opt.optimizer_type == "sparse_adam":
        sys.exit(f"Trying to use sparse adam but it is not installed, please install the correct rasterizer using pip install [3dgs_accel].")

    first_iter = 0
    tb_writer = prepare_output_and_logger(dataset)
    gaussians = GaussianModel(dataset.sh_degree, opt.optimizer_type)
    scene = Scene(dataset, gaussians)
    gaussians.training_setup(opt)
    if checkpoint:
        (model_params, first_iter) = torch.load(checkpoint)
        gaussians.restore(model_params, opt)

    bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
    background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

    iter_start = torch.cuda.Event(enable_timing = True)
    iter_end = torch.cuda.Event(enable_timing = True)

    use_sparse_adam = opt.optimizer_type == "sparse_adam" and SPARSE_ADAM_AVAILABLE 
    depth_l1_weight = get_expon_lr_func(opt.depth_l1_weight_init, opt.depth_l1_weight_final, max_steps=opt.iterations)

    viewpoint_stack = scene.getTrainCameras().copy()
    viewpoint_indices = list(range(len(viewpoint_stack)))
    ema_loss_for_log = 0.0
    ema_Ll1depth_for_log = 0.0

    progress_bar = tqdm(range(first_iter, opt.iterations), desc="Training progress")
    first_iter += 1
    for iteration in range(first_iter, opt.iterations + 1):
        if network_gui.conn == None:
            network_gui.try_connect()
        while network_gui.conn != None:
            try:
                net_image_bytes = None
                custom_cam, do_training, pipe.convert_SHs_python, pipe.compute_cov3D_python, keep_alive, scaling_modifer = network_gui.receive()
                if custom_cam != None:
                    net_image = render(custom_cam, gaussians, pipe, background, scaling_modifier=scaling_modifer, use_trained_exp=dataset.train_test_exp, separate_sh=SPARSE_ADAM_AVAILABLE)["render"]
                    net_image_bytes = memoryview((torch.clamp(net_image, min=0, max=1.0) * 255).byte().permute(1, 2, 0).contiguous().cpu().numpy())
                network_gui.send(net_image_bytes, dataset.source_path)
                if do_training and ((iteration < int(opt.iterations)) or not keep_alive):
                    break
            except Exception as e:
                network_gui.conn = None

        iter_start.record()

        gaussians.update_learning_rate(iteration)
        if iteration % 1000 == 0:
            print(f"\n[ITER {iteration}] Peak VRAM: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB")


        # Every 1000 its we increase the levels of SH up to a maximum degree
        if iteration % 1000 == 0:
            gaussians.oneupSHdegree()

        # Pick a random Camera
        if not viewpoint_stack:
            viewpoint_stack = scene.getTrainCameras().copy()
            viewpoint_indices = list(range(len(viewpoint_stack)))
        rand_idx = randint(0, len(viewpoint_indices) - 1)
        viewpoint_cam = viewpoint_stack.pop(rand_idx)
        vind = viewpoint_indices.pop(rand_idx)

        # Render
        if (iteration - 1) == debug_from:
            pipe.debug = True

        bg = torch.rand((3), device="cuda") if opt.random_background else background

        render_pkg = render(viewpoint_cam, gaussians, pipe, bg, use_trained_exp=dataset.train_test_exp, separate_sh=SPARSE_ADAM_AVAILABLE)
        image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]

        if viewpoint_cam.alpha_mask is not None:
            alpha_mask = viewpoint_cam.alpha_mask.cuda()
            image *= alpha_mask

        # Loss
        gt_image = viewpoint_cam.original_image.cuda()
        Ll1 = l1_loss(image, gt_image)
        if FUSED_SSIM_AVAILABLE:
            ssim_value = fused_ssim(image.unsqueeze(0), gt_image.unsqueeze(0))
        else:
            ssim_value = ssim(image, gt_image)

        loss = (1.0 - opt.lambda_dssim) * Ll1 + opt.lambda_dssim * (1.0 - ssim_value)

        # Depth regularization
        Ll1depth_pure = 0.0
        if depth_l1_weight(iteration) > 0 and viewpoint_cam.depth_reliable:
            invDepth = render_pkg["depth"]
            mono_invdepth = viewpoint_cam.invdepthmap.cuda()
            depth_mask = viewpoint_cam.depth_mask.cuda()

            Ll1depth_pure = torch.abs((invDepth  - mono_invdepth) * depth_mask).mean()
            Ll1depth = depth_l1_weight(iteration) * Ll1depth_pure 
            loss += Ll1depth
            Ll1depth = Ll1depth.item()
        else:
            Ll1depth = 0

        loss.backward()

        iter_end.record()

        with torch.no_grad():
            # Progress bar
            ema_loss_for_log = 0.4 * loss.item() + 0.6 * ema_loss_for_log
            ema_Ll1depth_for_log = 0.4 * Ll1depth + 0.6 * ema_Ll1depth_for_log

            if iteration % 10 == 0:
                progress_bar.set_postfix({"Loss": f"{ema_loss_for_log:.{7}f}", "Depth Loss": f"{ema_Ll1depth_for_log:.{7}f}"})
                progress_bar.update(10)
            if iteration == opt.iterations:
                progress_bar.close()

            # Log and save
            training_report(tb_writer, iteration, Ll1, loss, l1_loss, iter_start.elapsed_time(iter_end), testing_iterations, scene, render, (pipe, background, 1., SPARSE_ADAM_AVAILABLE, None, dataset.train_test_exp), dataset.train_test_exp)
            if (iteration in saving_iterations):
                print("\n[ITER {}] Saving Gaussians".format(iteration))
                scene.save(iteration)

            # Densification
            if iteration < opt.densify_until_iter:
                # Keep track of max radii in image-space for pruning
                gaussians.max_radii2D[visibility_filter] = torch.max(gaussians.max_radii2D[visibility_filter], radii[visibility_filter])
                gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter)

                if iteration > opt.densify_from_iter and iteration % opt.densification_interval == 0:
                    size_threshold = 20 if iteration > opt.opacity_reset_interval else None
                    gaussians.densify_and_prune(opt.densify_grad_threshold, 0.005, scene.cameras_extent, size_threshold, radii)
                
                if iteration % opt.opacity_reset_interval == 0 or (dataset.white_background and iteration == opt.densify_from_iter):
                    gaussians.reset_opacity()

            # Optimizer step
            if iteration < opt.iterations:
                gaussians.exposure_optimizer.step()
                gaussians.exposure_optimizer.zero_grad(set_to_none = True)
                if use_sparse_adam:
                    visible = radii > 0
                    gaussians.optimizer.step(visible, radii.shape[0])
                    gaussians.optimizer.zero_grad(set_to_none = True)
                else:
                    gaussians.optimizer.step()
                    gaussians.optimizer.zero_grad(set_to_none = True)

            if (iteration in checkpoint_iterations):
                print("\\n[ITER {}] Saving Checkpoint".format(iteration))
                save_path = scene.model_path + "/chkpnt" + str(iteration) + ".pth"
                torch.save((gaussians.capture(), iteration), save_path)
                import shutil
                if os.path.exists(os.environ.get('KAGGLE_WORKING_DIR', '/kaggle/working/')):
                    shutil.copy(save_path, os.environ.get('KAGGLE_WORKING_DIR', '/kaggle/working/'))

def prepare_output_and_logger(args):    
    if not args.model_path:
        if os.getenv('OAR_JOB_ID'):
            unique_str=os.getenv('OAR_JOB_ID')
        else:
            unique_str = str(uuid.uuid4())
        args.model_path = os.path.join("./output/", unique_str[0:10])
        
    # Set up output folder
    print("Output folder: {}".format(args.model_path))
    os.makedirs(args.model_path, exist_ok = True)
    with open(os.path.join(args.model_path, "cfg_args"), 'w') as cfg_log_f:
        cfg_log_f.write(str(Namespace(**vars(args))))

    # Create Tensorboard writer
    tb_writer = None
    if TENSORBOARD_FOUND:
        tb_writer = SummaryWriter(args.model_path)
    else:
        print("Tensorboard not available: not logging progress")
    return tb_writer

def training_report(tb_writer, iteration, Ll1, loss, l1_loss, elapsed, testing_iterations, scene : Scene, renderFunc, renderArgs, train_test_exp):
    if tb_writer:
        tb_writer.add_scalar('train_loss_patches/l1_loss', Ll1.item(), iteration)
        tb_writer.add_scalar('train_loss_patches/total_loss', loss.item(), iteration)
        tb_writer.add_scalar('iter_time', elapsed, iteration)

    # Report test and samples of training set
    if iteration in testing_iterations:
        torch.cuda.empty_cache()
        validation_configs = ({'name': 'test', 'cameras' : scene.getTestCameras()}, 
                              {'name': 'train', 'cameras' : [scene.getTrainCameras()[idx % len(scene.getTrainCameras())] for idx in range(5, 30, 5)]})

        for config in validation_configs:
            if config['cameras'] and len(config['cameras']) > 0:
                l1_test = 0.0
                psnr_test = 0.0
                for idx, viewpoint in enumerate(config['cameras']):
                    image = torch.clamp(renderFunc(viewpoint, scene.gaussians, *renderArgs)["render"], 0.0, 1.0)
                    gt_image = torch.clamp(viewpoint.original_image.to("cuda"), 0.0, 1.0)
                    if train_test_exp:
                        image = image[..., image.shape[-1] // 2:]
                        gt_image = gt_image[..., gt_image.shape[-1] // 2:]
                    if tb_writer and (idx < 5):
                        tb_writer.add_images(config['name'] + "_view_{}/render".format(viewpoint.image_name), image[None], global_step=iteration)
                        if iteration == testing_iterations[0]:
                            tb_writer.add_images(config['name'] + "_view_{}/ground_truth".format(viewpoint.image_name), gt_image[None], global_step=iteration)
                    l1_test += l1_loss(image, gt_image).mean().double()
                    psnr_test += psnr(image, gt_image).mean().double()
                psnr_test /= len(config['cameras'])
                l1_test /= len(config['cameras'])          
                print("\n[ITER {}] Evaluating {}: L1 {} PSNR {}".format(iteration, config['name'], l1_test, psnr_test))
                if tb_writer:
                    tb_writer.add_scalar(config['name'] + '/loss_viewpoint - l1_loss', l1_test, iteration)
                    tb_writer.add_scalar(config['name'] + '/loss_viewpoint - psnr', psnr_test, iteration)

        if tb_writer:
            tb_writer.add_histogram("scene/opacity_histogram", scene.gaussians.get_opacity, iteration)
            tb_writer.add_scalar('total_points', scene.gaussians.get_xyz.shape[0], iteration)
        torch.cuda.empty_cache()

if __name__ == "__main__":
    # Set up command line argument parser
    parser = ArgumentParser(description="Training script parameters")
    lp = ModelParams(parser)
    op = OptimizationParams(parser)
    pp = PipelineParams(parser)
    parser.add_argument('--ip', type=str, default="127.0.0.1")
    parser.add_argument('--port', type=int, default=6009)
    parser.add_argument('--debug_from', type=int, default=-1)
    parser.add_argument('--detect_anomaly', action='store_true', default=False)
    parser.add_argument("--test_iterations", nargs="+", type=int, default=[7_000, 30_000])
    parser.add_argument("--save_iterations", nargs="+", type=int, default=[7_000, 30_000])
    parser.add_argument("--quiet", action="store_true")
    parser.add_argument('--disable_viewer', action='store_true', default=False)
    parser.add_argument("--checkpoint_iterations", nargs="+", type=int, default=[])
    parser.add_argument("--start_checkpoint", type=str, default = None)
    parser.add_argument("--kaggle_working_dir", type=str, default='/kaggle/working/')
    parser.add_argument("--resume", action='store_true', help='Auto-resume from latest checkpoint in model_path')
    args = parser.parse_args(sys.argv[1:])
    args.save_iterations.append(args.iterations)
    
    
    # Resume logic
    if args.resume and not args.start_checkpoint:
        import glob
        chkpts = glob.glob(os.path.join(args.model_path, "chkpnt*.pth"))
        if len(chkpts) > 0:
            # Sort by iteration number
            chkpts.sort(key=lambda x: int(re.findall(r"chkpnt(\d+).pth", x)[0]))
            args.start_checkpoint = chkpts[-1]
            print(f"Resuming from {args.start_checkpoint}")

    print("Optimizing " + args.model_path)

    # Initialize system state (RNG)
    safe_state(args.quiet)

    # Start GUI server, configure and run training
    if not args.disable_viewer:
        network_gui.init(args.ip, args.port)
    torch.autograd.set_detect_anomaly(args.detect_anomaly)
    training(lp.extract(args), op.extract(args), pp.extract(args), args.test_iterations, args.save_iterations, args.checkpoint_iterations, args.start_checkpoint, args.debug_from)

    # All done
    print("\nTraining complete.")
    if os.path.exists(args.kaggle_working_dir):
        import zipfile
        out_zip = os.path.join(args.kaggle_working_dir, "training_run.zip")
        print(f"Zipping outputs to {out_zip}")
        with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(args.model_path):
                for f in files:
                    # Zip everything in the output directory
                    abs_path = os.path.join(root, f)
                    rel_path = os.path.relpath(abs_path, args.model_path)
                    zipf.write(abs_path, rel_path)


In [ ]:

# 5. Point to dataset at /kaggle/input/<dataset-name>/, print its structure to confirm visibility.
import os
dataset_path = "/kaggle/input/<dataset-name>/"
print(f"Dataset path: {dataset_path}")
if os.path.exists(dataset_path):
    print("Dataset structure:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print('{}{}/'.format(indent, os.path.basename(root)))
        subindent = ' ' * 4 * (level + 1)
        for f in files[:3]:
            print('{}{}'.format(subindent, f))
        if len(files) > 3:
            print('{}{}... and {} more files'.format(subindent, len(files)-3))
else:
    print("Dataset directory not found on Kaggle! Please attach the dataset.")


In [ ]:

# 6. Sanity-check import of my repo's loader/model classes (no full training).
import sys
# Make sure we can import from the repo
if '/kaggle/working/repo' not in sys.path:
    sys.path.append('/kaggle/working/repo')

# The user's repo is a fork/clone of gaussian-splatting
from scene.dataset_readers import readColmapSceneInfo
from scene.gaussian_model import GaussianModel
print("Successfully imported readColmapSceneInfo and GaussianModel!")


In [ ]:

# 7. Placeholder cell for training with commented-out args to fill in.
# !python train.py -s /kaggle/input/<dataset-name>/data/HCM0421 --iterations 30000 --eval
print("Uncomment the line above and set the correct dataset path to start training.")


In [ ]:

# 8. Placeholder cell for rendering test poses + zipping submission.zip.
# !python render_test_poses.py -m output/xxxx --test_poses_csv /kaggle/input/<dataset-name>/data/HCM0421/test/test_poses.csv --output_dir renders/
# !python package_submission.py
print("Uncomment the lines above to render test poses and package the submission.zip.")
